# 資料前處理

In [1]:
import pandas as pd
import numpy as np
train_data = pd.read_csv("train_info.csv")
test_data = pd.read_csv("test_info.csv")

In [2]:
import os

train_dict = {}

for i in train_data["unique_id"].unique():
    file_path = os.path.join("./train_data", f"{i}.txt")
    #with open(file_path, "r", encoding="utf-8") as f:
    #    lines = f.readlines()

    # parse each line into list of numbers
    #data = [list(map(int, line.strip().split())) for line in lines if line.strip()]
    #train_dict[i] = data
    
    with open(file_path, "r", encoding="utf-8") as f:
        # split all numbers across lines, flatten into one list
        numbers = [int(x) for line in f for x in line.strip().split()]
    train_dict[i] = numbers[:102]

    
test_dict = {}
for i in test_data["unique_id"].unique():
    file_path = os.path.join("./test_data", f"{i}.txt")
    
    with open(file_path, "r", encoding="utf-8") as f:
        # split all numbers across lines, flatten into one list
        numbers = [int(x) for line in f for x in line.strip().split()]
    test_dict[i] = numbers[:102]

    

In [3]:
#import predict goals
train_gender = train_data["gender"]
train_hand = train_data["hold racket handed"]
train_level = train_data["level"]
train_years = train_data["play years"]

test_ans = pd.read_csv("test_answer.csv")
test_gender = test_ans["gender"]
test_hand = test_ans["hold racket handed"]
test_level = test_ans["level"]
test_years = test_ans["play years"]


# Torch

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

In [25]:
X = torch.tensor(list(train_dict.values()), dtype=torch.float32)#.unsqueeze(0)
y = torch.tensor(train_level,dtype=torch.long) -1 #.unsqueeze(0)

In [27]:
y

tensor([4, 4, 4,  ..., 3, 3, 3])

# MLP

In [22]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x  # 如果分類，通常搭配 CrossEntropyLoss

In [23]:
#調整參數
model = MLP(input_size=102, hidden_size=16, output_size=5)
criterion = nn.CrossEntropyLoss()  # 分類任務
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [28]:
# 訓練 loop
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()

方法二 但沒有比較快？

In [130]:
from torch.utils.data import TensorDataset, DataLoader

# 同樣資料
dataset = TensorDataset(X, y)

batch_size = 2
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

model = MLP(input_size=24, hidden_size=16, output_size=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

print("Mini-batch training done!")

Mini-batch training done!


Predict

In [29]:
#Predict data:
test_X = torch.tensor(list(test_dict.values()), dtype=torch.float32)
test_y = torch.tensor(test_level,dtype=torch.float32)-1

In [30]:
#test_seq = torch.tensor([[[4], [5], [6], [7], [8]]], dtype=torch.float32)
pred = model(test_X)
pred_classes = torch.argmax(pred)
#print("🔮 預測結果:", pred.item())

In [20]:
pred

tensor([[   56.5506,   104.5810],
        [ -704.0015,  -889.3743],
        [ -586.9965,  -954.7288],
        ...,
        [ 4717.7959,  3783.6357],
        [ -224.4671,  -288.4875],
        [-1027.4205, -1317.7616]], grad_fn=<AddmmBackward0>)

In [31]:
pred_classes = torch.argmax(pred, dim=1)
print(pred_classes)

tensor([2, 4, 2,  ..., 2, 4, 4])


# 算成績

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [32]:
pred_np = pred_classes.numpy()
true_np = test_y.numpy()

In [1]:
accuracy = accuracy_score(true_np, pred_np)
precision = precision_score(true_np, pred_np, average='macro')
recall = recall_score(true_np, pred_np,average='macro')
f1 = f1_score(true_np, pred_np,average='macro')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

NameError: name 'accuracy_score' is not defined